# Microsoft Agent Framework — Azure OpenAI (API de Respuestas)

En este ejemplo de código, usarás el **Microsoft Agent Framework (MAF)** para crear un agente simple respaldado por **Azure OpenAI** utilizando la **API de Respuestas**.

> **Nota de migración:** Este ejemplo utilizaba previamente Semantic Kernel con GitHub Models. Ha sido migrado al Microsoft Agent Framework, y GitHub Models (obsoleto, retirándose en julio de 2026) ha sido reemplazado por Azure OpenAI, que soporta la API de Respuestas. El `OpenAIChatClient` en MAF apunta al endpoint estable `/openai/v1/` de Azure OpenAI y utiliza la API de Respuestas por defecto.

El propósito de este ejemplo es demostrar los pasos que luego se aplicarán en ejemplos de código adicionales al implementar varios patrones agenticos.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importa los Paquetes de Python Necesarios


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definiendo una Herramienta

En el Microsoft Agent Framework, una **herramienta** es una función Python sencilla decorada con `@tool` que el agente puede llamar. A continuación definimos una herramienta que devuelve un destino de vacaciones aleatorio y evita repetir el anterior.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Creando el Agente

Aquí, creamos el Agente llamado `TravelAgent`.

En este ejemplo, usamos instrucciones muy básicas. Siéntase libre de modificar estas instrucciones para observar cómo cambia el comportamiento del agente.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Ejecutando el Agente

Ahora podemos ejecutar el agente. Creamos una `AgentSession` para que el agente recuerde la conversación a lo largo de los turnos, luego enviamos dos `user_inputs`. El primero pide un viaje; el segundo dice que al usuario no le gustó la sugerencia y pide otra — el agente usa el historial de la sesión más la herramienta `get_random_destination` para responder.

Puedes modificar estos mensajes para observar cómo reacciona el agente de manera diferente. Las respuestas se **transmiten** token por token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
